# 04 Connectivity plots and QC

This notebook provides generic QC and first descriptive plots for source-label connectivity outputs.

It is intentionally project-neutral. Set `FOCUS_TASK`, `FOCUS_CONDITION`, `PRE_WINDOW`, and `POST_WINDOW` below to inspect a specific subset or contrast. Leave values as `None` to use broad/default selections from the available connectivity files.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from meeg_pipeline.config import load_config
from meeg_pipeline.connectivity import (
    connectivity_matrix_summary_to_dataframe,
    connectivity_qc_to_dataframe,
    load_connectivity_npz,
)


def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for path in [start, *start.parents]:
        if (path / "configs/local.yaml").exists():
            return path
    raise FileNotFoundError("Could not find project root containing configs/local.yaml")


PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "configs/local.yaml"
FIGURE_DIR = PROJECT_ROOT / "derivatives/meeg-pipeline/qc/connectivity"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

config = load_config(CONFIG_PATH)

# Optional focus settings. Set to strings from your config/output files, or leave as None.
FOCUS_TASK = None
FOCUS_CONDITION = None
PRE_WINDOW = None
POST_WINDOW = None

print("Project root:", PROJECT_ROOT)
print("Connectivity QC figure directory:", FIGURE_DIR)
print("Focus task:", FOCUS_TASK)
print("Focus condition:", FOCUS_CONDITION)
print("Window contrast:", PRE_WINDOW, "->", POST_WINDOW)


## QC table

This table checks whether connectivity outputs exist and were written successfully.


In [ ]:
qc = connectivity_qc_to_dataframe(config)
print("QC shape:", qc.shape)

if qc.empty:
    print("No connectivity QC rows found.")
else:
    display(qc)
    print(qc["status"].value_counts(dropna=False).to_string())

    focus_qc = qc.copy()
    if FOCUS_TASK is not None and "task" in focus_qc.columns:
        focus_qc = focus_qc[focus_qc["task"].astype(str).str.lower() == FOCUS_TASK.lower()]
    if FOCUS_CONDITION is not None and "condition" in focus_qc.columns:
        focus_qc = focus_qc[focus_qc["condition"].astype(str).str.lower() == FOCUS_CONDITION.lower()]
    if PRE_WINDOW is not None and POST_WINDOW is not None and "window" in focus_qc.columns:
        focus_qc = focus_qc[focus_qc["window"].isin([PRE_WINDOW, POST_WINDOW])]

    print("Focused QC shape:", focus_qc.shape)
    display(focus_qc)


## Matrix-level summaries

These descriptive summaries help spot outliers, method differences, or unexpectedly empty matrices.


In [ ]:
summary = connectivity_matrix_summary_to_dataframe(config)
print("Summary shape:", summary.shape)

wanted = [
    "subject", "task", "method", "window", "condition", "band",
    "n_epochs", "n_labels", "mean", "mean_abs", "max_abs", "status",
]
cols = [col for col in wanted if col in summary.columns]
display(summary[cols] if cols else summary)

focus_summary = summary.copy()
if not focus_summary.empty:
    focus_summary = focus_summary.query("status == 'ok'").copy() if "status" in focus_summary.columns else focus_summary
    if FOCUS_TASK is not None and "task" in focus_summary.columns:
        focus_summary = focus_summary[focus_summary["task"].astype(str).str.lower() == FOCUS_TASK.lower()]
    if FOCUS_CONDITION is not None and "condition" in focus_summary.columns:
        focus_summary = focus_summary[focus_summary["condition"].astype(str).str.lower() == FOCUS_CONDITION.lower()]

print("Focused summary shape:", focus_summary.shape)
display(focus_summary[cols] if cols else focus_summary)


## Load connectivity files

Build a table from all `*-con.npz` files in the pipeline derivatives tree.


In [ ]:
def path_table(paths):
    rows = []
    for path in paths:
        loaded = load_connectivity_npz(path)
        name = Path(path).name
        subject = next((part for part in name.split("_") if part.startswith("sub-")), None)
        task = next((part.removeprefix("task-") for part in name.split("_") if part.startswith("task-")), None)
        rows.append(
            {
                "subject": subject,
                "task": task,
                "method": loaded["method"],
                "window": loaded["window"],
                "condition": loaded["condition"],
                "path": str(path),
            }
        )
    return pd.DataFrame(rows)


paths = sorted(Path(config.paths.derivatives_root).glob("sub-*/*/connectivity/*-con.npz"))
if not paths:
    paths = sorted(Path(config.paths.derivatives_root).glob("sub-*/connectivity/*-con.npz"))
print(f"Found {len(paths)} connectivity files")

pt = path_table(paths) if paths else pd.DataFrame()
display(pt)

focus_pt = pt.copy()
if not focus_pt.empty:
    if FOCUS_TASK is not None:
        focus_pt = focus_pt[focus_pt["task"].astype(str).str.lower() == FOCUS_TASK.lower()]
    if FOCUS_CONDITION is not None:
        focus_pt = focus_pt[focus_pt["condition"].astype(str).str.lower() == FOCUS_CONDITION.lower()]
    if PRE_WINDOW is not None and POST_WINDOW is not None:
        focus_pt = focus_pt[focus_pt["window"].isin([PRE_WINDOW, POST_WINDOW])]
    focus_pt = focus_pt.sort_values([col for col in ["subject", "method", "window"] if col in focus_pt.columns])

print("Focused files:", len(focus_pt))
display(focus_pt)


## Single-file heatmap for QC

This is a sanity check for one connectivity matrix. Adjust the method, band, and window variables as needed.


In [ ]:
HEATMAP_METHOD = None
HEATMAP_BAND = None
HEATMAP_WINDOW = POST_WINDOW

heatmap_candidates = focus_pt.copy() if not focus_pt.empty else pd.DataFrame()
if not heatmap_candidates.empty and HEATMAP_METHOD is not None:
    heatmap_candidates = heatmap_candidates[heatmap_candidates["method"] == HEATMAP_METHOD]
if not heatmap_candidates.empty and HEATMAP_WINDOW is not None:
    heatmap_candidates = heatmap_candidates[heatmap_candidates["window"] == HEATMAP_WINDOW]

if heatmap_candidates.empty:
    print("No heatmap candidate found. Set focus variables or run connectivity first.")
else:
    heatmap_path = heatmap_candidates.iloc[0]["path"]
    loaded = load_connectivity_npz(heatmap_path)
    band_idx = 0 if HEATMAP_BAND is None else loaded["band_names"].index(HEATMAP_BAND)
    matrix = loaded["connectivity"][:, :, band_idx]
    labels = loaded["labels"]

    fig, ax = plt.subplots(figsize=(8, 7))
    vmax = np.nanmax(np.abs(matrix))
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1.0
    im = ax.imshow(matrix, aspect="auto", interpolation="nearest", vmin=-vmax, vmax=vmax)
    ax.set_title(f"QC single matrix · {loaded['method']} · {loaded['band_names'][band_idx]} · {loaded['window']}")
    ax.set_xlabel("Label to")
    ax.set_ylabel("Label from")
    step = max(1, len(labels) // 12)
    ticks = np.arange(0, len(labels), step)
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)
    ax.set_xticklabels([labels[i] for i in ticks], rotation=90, fontsize=7)
    ax.set_yticklabels([labels[i] for i in ticks], fontsize=7)
    fig.colorbar(im, ax=ax, shrink=0.8)
    fig.tight_layout()
    out = FIGURE_DIR / f"qc_heatmap_{Path(loaded['path']).stem}_{loaded['band_names'][band_idx]}.png"
    fig.savefig(out, dpi=150)
    display(fig)
    plt.close(fig)
    print("saved", out)


## Optional window contrast

If `PRE_WINDOW` and `POST_WINDOW` are set, compute `POST_WINDOW - PRE_WINDOW` per subject and average the contrast matrices.


In [ ]:
def compute_window_contrasts(path_df: pd.DataFrame, method: str, band: str) -> tuple[list[np.ndarray], dict | None]:
    diffs = []
    meta = None
    if path_df.empty or PRE_WINDOW is None or POST_WINDOW is None:
        return diffs, meta

    sub_df = path_df[path_df["method"] == method]
    for subject, grp in sub_df.groupby("subject"):
        by_window = {row.window: row.path for row in grp.itertuples()}
        if PRE_WINDOW not in by_window or POST_WINDOW not in by_window:
            print(f"Skipping {subject}: missing {PRE_WINDOW} or {POST_WINDOW}")
            continue

        pre = load_connectivity_npz(by_window[PRE_WINDOW])
        post = load_connectivity_npz(by_window[POST_WINDOW])
        if band not in pre["band_names"] or band not in post["band_names"]:
            print(f"Skipping {subject}: band {band} missing")
            continue

        diff = post["connectivity"][:, :, post["band_names"].index(band)] - pre["connectivity"][:, :, pre["band_names"].index(band)]
        diffs.append(diff)
        if meta is None:
            meta = {
                "labels": post["labels"],
                "method": method,
                "band": band,
                "condition": post["condition"],
            }
    return diffs, meta


if PRE_WINDOW is None or POST_WINDOW is None or focus_pt.empty:
    print("Set PRE_WINDOW and POST_WINDOW and ensure connectivity files exist to compute contrasts.")
else:
    contrast_rows = []
    for method in sorted(focus_pt["method"].dropna().unique()):
        candidate = load_connectivity_npz(focus_pt[focus_pt["method"] == method].iloc[0]["path"])
        for band in candidate["band_names"]:
            diffs, meta = compute_window_contrasts(focus_pt, method=method, band=band)
            if not diffs or meta is None:
                continue
            stack = np.stack(diffs, axis=0)
            mean_diff = np.nanmean(stack, axis=0)
            contrast_rows.append(
                {
                    "condition": meta["condition"],
                    "method": method,
                    "band": band,
                    "n_subjects": stack.shape[0],
                    "mean": float(np.nanmean(mean_diff)),
                    "mean_abs": float(np.nanmean(np.abs(mean_diff))),
                    "max_abs": float(np.nanmax(np.abs(mean_diff))),
                }
            )
    contrast_df = pd.DataFrame(contrast_rows)
    display(contrast_df)
